# Chapter 1 &mdash; Context-Free Patterns: Nesting and Palindromes

**Concept 11 of the Chapter 1 decomposition:** *Pattern Class II -- Context-Free Patterns*

Proper nesting needs a running count with no bound &mdash; exactly what a <b>stack</b> gives you, and what finite memory cannot.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Context-Free-Patterns/Concept-Context-Free-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimatePDA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimatePDA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Properly nested parentheses have two properties:

* the same, finite-but-unbounded, number of `(` and `)`;
* sweeping left to right, the count of `(` is **at every point** at least the count
  of `)`, with equality at the end.

That second condition demands an unbounded running count. A **palindrome** is the
other archetype: it needs the front matched against the *reverse* of the back.

Both are what a last-in-first-out memory buys you.

## 2. Definitions

### A PDA for the Dyck language

Push on `(`, pop on `)`. The stack **is** the running count.
The edge label reads `input , pop ; push`.

In [ ]:
dyck = md2mc('''PDA
IF : ( , # ; (#  -> M
M  : ( , ( ; ((  -> M
M  : ) , ( ; ''  -> M
M  : '' , # ; #  -> IF
''')
print("Dyck PDA states :", sorted(dyck["Q"]))

### A stack checker in plain Python

The same algorithm, so you can see what the PDA is doing.

In [ ]:
def dyck_ok(s):
    depth = 0
    for ch in s:
        depth += 1 if ch == '(' else -1
        if depth < 0:          # a ')' arrived with nothing to match
            return False
    return depth == 0

### Palindromes: $ww^R$ versus a copy

Reversal is cheap for a stack; copying is not. Hold onto this &mdash; Concept 12 turns on it.

In [ ]:
def is_palindrome(s):
    return s == s[::-1]

## 3. Tests

The nesting checker, on the book's examples.

In [ ]:
for s in ['', '()', '(())', '(()(()))', '()()', ')(', '(()', '())(']:
    print("%-10s properly nested? %s" % (repr(s), dyck_ok(s)))
assert dyck_ok("(()(()))") and not dyck_ok(")(") and not dyck_ok("(()")

Run the **PDA** on the same strings. `explore_pda` prints every accepting run;
`STKMAX` bounds the stack during the search.

In [ ]:
explore_pda("(())", dyck, STKMAX=6)

Palindromes, and the crucial contrast.

In [ ]:
for s in ['0110', '010', '', '0101']:
    print("%-6s palindrome? %-6s  (is it w w^R? %s)"
          % (repr(s), is_palindrome(s), s == s[::-1]))
print()
print("w w^R (palindrome) : CONTEXT-FREE -- a stack returns things reversed.")
print("w w   (a copy)     : NOT context-free -- see Concept 12.")

## 4. Animation


Watch the stack grow and shrink as the PDA reads the string. The **height of the
stack is the nesting depth** &mdash; the unbounded count a DFA could not keep.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(dyck, FuseEdges=False)

## 5. Exercises


1. Run `dyck_ok` on `'(()'` and on `'())('`. Both fail &mdash; but for *different*
   reasons. Which clause of the definition does each violate?
2. Build a PDA for balanced `[` and `]` **mixed with** `(` and `)`, where the kinds
   must match. What extra stack symbols do you need?
3. Find the midpoint of the book's long palindrome by counting `0`s between `1`s.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter1/Concept-Context-Free-Patterns')